# Project Knowledge Base Retrieval System (RAG)

By Akshat Kesharwani | Free tools only (Google Colab + free-tier APIs)

## Problem Statement

Big companies have tons of internal docs that are hard to search with plain keywords. RAG (Retrieval-Augmented Generation) fixes this -- it finds the most relevant chunks of text first, then feeds them to an LLM so the answer is actually grounded in your documents instead of the model just guessing. Here we build a full RAG pipeline using free tools end to end.

### SECTION 1: Install & Import Libraries

In [1]:
!pip install -q chromadb sentence-transformers datasets rank_bm25 openai

import os
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from datasets import load_dataset, Dataset
from rank_bm25 import BM25Okapi

# ---- Small AI helper: call Groq (free) ----
# Only using Groq here -- it's free, fast, and just needs an API key,
# no extra setup like Ollama needs.
# Get a free key at https://console.groq.com/keys
# (Update: as of June 2026 Groq retired the old Llama-3.3-70B model,
# the current one to use is "openai/gpt-oss-120b".)
from getpass import getpass
from openai import OpenAI

groq_key = getpass("Enter your Groq API key: ")
groq_client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=groq_key)

def call_llm(prompt, max_tokens=300):
    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens
    )
    return response.choices[0].message.content

Enter your Groq API key: ··········


### SECTION 2: Load / Build an Enterprise-Style Document Corpus (robust fallback)

In [2]:
# heads up: the old "wikipedia" repo config ("20220301.simple") used a
# loading script and doesn't work anymore. HuggingFace's proper, actively
# maintained version is "wikimedia/wikipedia", which only has "20231101.*"
# configs now. we try that first, then fall back to small built-in
# synthetic documents. Wikipedia is only a free stand-in for internal
# enterprise docs -- the same chunk/embed/retrieve pipeline applies
# directly to your own policies, wikis, or runbooks.
def robust_load_wiki():
    candidates = [
        ("wikimedia/wikipedia", "20231101.simple"),
        ("wikimedia/wikipedia", "20231101.en"),
    ]
    for repo_id, config in candidates:
        try:
            print(f"[dataset] Trying load_dataset('{repo_id}', '{config}')...")
            ds = load_dataset(repo_id, config, split="train")
            print(f"[dataset] Loaded '{repo_id}' ({config}) successfully ({len(ds)} rows).")
            return ds
        except Exception as e:
            print(f"[dataset] '{repo_id}' ({config}) failed: {type(e).__name__}: {e}")
        try:
            parquet_path = f"hf://datasets/{repo_id}@refs/convert/parquet/{config}/train/*.parquet"
            print(f"[dataset] Trying Parquet fallback: {parquet_path}")
            ds = load_dataset("parquet", data_files=parquet_path, split="train")
            print(f"[dataset] Parquet fallback succeeded ({len(ds)} rows).")
            return ds
        except Exception as e:
            print(f"[dataset] Parquet fallback failed: {type(e).__name__}: {e}")
    return None

raw = robust_load_wiki()

if raw is not None:
    docs = raw.select(range(min(3000, len(raw))))
    corpus = [{"id": str(i), "text": d["text"][:1500], "title": d["title"]} for i, d in enumerate(docs)]
else:
    # ---- Synthetic offline fallback -- guarantees the notebook always runs ----
    SYNTHETIC_DOCS = [
        {"title": "Remote Work Policy", "text": "Employees may work remotely up to 3 days per week, subject to manager approval. Core collaboration hours are 10 AM to 3 PM local time. All remote work requires a stable internet connection and access to the company VPN."},
        {"title": "Expense Reimbursement", "text": "Business expenses under $75 do not require pre-approval. Receipts must be submitted within 30 days via the expense portal. Travel, meals, and client entertainment are reimbursable; personal purchases are not."},
        {"title": "Data Retention Policy", "text": "Customer data is retained for 7 years after account closure to meet regulatory requirements. Logs and analytics data are retained for 90 days. Requests for early deletion must go through the compliance team."},
        {"title": "Incident Response Runbook", "text": "On detecting a production incident, page the on-call engineer immediately. Severity 1 incidents require a status page update within 15 minutes. A postmortem must be published within 5 business days of resolution."},
        {"title": "Onboarding Checklist", "text": "New hires receive laptop and account provisioning on day one. IT security training must be completed within the first week. A 30-60-90 day plan is reviewed with the manager at each milestone."},
        {"title": "Paris, France", "text": "Paris is the capital and most populous city of France. It is known for landmarks such as the Eiffel Tower, the Louvre Museum, and Notre-Dame Cathedral. The city sits on the Seine River in northern France."},
        {"title": "Albert Einstein", "text": "Albert Einstein was a theoretical physicist who developed the theory of relativity, one of the two pillars of modern physics. He received the Nobel Prize in Physics in 1921 for his explanation of the photoelectric effect."},
        {"title": "Photosynthesis", "text": "Photosynthesis is the process by which green plants and some organisms convert light energy into chemical energy. It occurs mainly in the chloroplasts of plant cells and produces oxygen as a byproduct."},
    ]
    corpus = [{"id": str(i), "text": d["text"], "title": d["title"]} for i, d in enumerate(SYNTHETIC_DOCS)]
    print("[dataset] Using synthetic fallback: swap in your own enterprise documents before reporting final metrics.")

print("Corpus size:", len(corpus))

[dataset] Trying load_dataset('wikimedia/wikipedia', '20231101.simple')...


Generating train split:   0%|          | 0/241787 [00:00<?, ? examples/s]

[dataset] Loaded 'wikimedia/wikipedia' (20231101.simple) successfully (241787 rows).
Corpus size: 3000


### SECTION 3: Chunk Documents

In [3]:
def chunk_text(text, chunk_size=300, overlap=50):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunks.append(" ".join(words[i:i + chunk_size]))
    return chunks

chunked_corpus = []
for doc in corpus:
    doc_id = doc["id"]
    for j, chunk in enumerate(chunk_text(doc["text"])):
        chunked_corpus.append({"id": f"{doc_id}_{j}", "text": chunk, "title": doc["title"]})

print("Total chunks:", len(chunked_corpus))

Total chunks: 4040


### SECTION 4: Embed Chunks with MPNet

In [4]:
embedding_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
texts = [c["text"] for c in chunked_corpus]
embeddings = embedding_model.encode(texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/64 [00:00<?, ?it/s]

### SECTION 5: Store in ChromaDB

In [5]:
client = chromadb.PersistentClient(path="./chroma_kb")
collection = client.get_or_create_collection("enterprise_kb")

collection.add(
    ids=[c["id"] for c in chunked_corpus],
    embeddings=embeddings.tolist(),
    documents=texts,
    metadatas=[{"title": c["title"]} for c in chunked_corpus]
)
print("Indexed", collection.count(), "chunks into ChromaDB")

Indexed 4040 chunks into ChromaDB


### SECTION 6: Hybrid Retrieval (Dense + BM25)

In [6]:
tokenized_corpus = [t.split() for t in texts]
bm25 = BM25Okapi(tokenized_corpus)

def retrieve(query, top_k=5, alpha=0.6):
    # dense retrieval
    query_emb = embedding_model.encode([query])[0].tolist()
    dense_results = collection.query(query_embeddings=[query_emb], n_results=top_k * 2)
    dense_ids = dense_results["ids"][0]
    dense_docs = dense_results["documents"][0]

    # sparse (BM25) retrieval for keyword coverage
    bm25_scores = bm25.get_scores(query.split())
    bm25_top_idx = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:top_k * 2]
    bm25_docs = [texts[i] for i in bm25_top_idx]

    # merge, dense-first, dedup
    merged = list(dict.fromkeys(dense_docs + bm25_docs))
    return merged[:top_k]

### SECTION 7: Measure Retrieval Precision

In [7]:
# Simple eval: for a set of sample queries with known relevant titles,
# check what fraction of top-k retrieved chunks come from the right doc.
sample_queries = [
    ("What is the capital of France?", "Paris"),
    ("Who wrote the theory of relativity?", "Albert Einstein"),
]

def precision_at_k(query, expected_title, k=5):
    results = collection.query(query_embeddings=[embedding_model.encode([query])[0].tolist()], n_results=k)
    titles = [m["title"] for m in results["metadatas"][0]]
    hits = sum(1 for t in titles if expected_title.lower() in t.lower())
    return hits / k

precisions = [precision_at_k(q, t) for q, t in sample_queries]
print(f"Average retrieval precision@5 on sample queries: {sum(precisions)/len(precisions):.2f}")

Average retrieval precision@5 on sample queries: 0.20


### SECTION 8: Generate Answer with Retrieved Context (RAG)

In [8]:
# using our call_llm() helper from earlier (Groq) to
# actually write the answer, grounded in the chunks we retrieved above.
def rag_answer(query, top_k=3):
    context_chunks = retrieve(query, top_k=top_k)
    context = "\n\n".join(context_chunks)
    prompt = f"Answer the question using ONLY the context below.\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:"
    return call_llm(prompt, max_tokens=200)

answer = rag_answer("What is the capital of France?")
print("RAG answer:", answer)

RAG answer: Paris.
